# Zero-shot: mixed inverted-Mel → LA2019

Score the **2017+PA** inverted-Mel checkpoint on ASVspoof **2019 LA** (no fine-tune).
Expect weak transfer — this is the baseline before an LA-specific model.

In [1]:
from pathlib import Path
import csv
import json
import sys

import numpy as np
import torch
from IPython.display import Markdown, display
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / "la_data.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\inverted_mel_mixed_on_la2019")

REPO = ROOT
for _ in range(6):
    if (REPO / "data" / "LA").exists():
        break
    REPO = REPO.parent

INVERTED_MEL = ROOT.parent / "inverted_mel"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(INVERTED_MEL))

from inverted_mel_cnn import AudioConfig, ReplayCNN, fix_length
from la_data import (
    AUDIO_DIR_BY_SPLIT,
    PROTOCOL_BY_SPLIT,
    LAWaveformDataset,
    filter_readable_records,
    read_la_cm_protocol,
)

LA_ROOT = REPO / "data" / "LA"
CKPT = (
    ROOT.parent
    / "inverted_mel_mixed_2017_pa2019"
    / "runs"
    / "inverted_mel_mixed"
    / "best_inverted_mel_mixed_2017_pa2019.pt"
)
OUT = ROOT / "runs" / "zero_shot_la"
CACHE = ROOT / "cache"

print("LA_ROOT", LA_ROOT, "exists", LA_ROOT.exists())
print("CKPT   ", CKPT, "exists", CKPT.exists())
print("cuda   ", torch.cuda.is_available())

LA_ROOT d:\speaker-verification-system\data\LA exists True
CKPT    d:\speaker-verification-system\replay-cnn-baseline\experiments\inverted_mel_mixed_2017_pa2019\runs\inverted_mel_mixed\best_inverted_mel_mixed_2017_pa2019.pt exists True
cuda    True


## Knobs
Set `MAX_UTTS > 0` for a quick smoke test (e.g. 2000).

In [2]:
SPLIT = "dev"          # train | dev | eval
MAX_UTTS = 0           # 0 = all readable
BATCH_SIZE = 8
WORKERS = 0
FORCE_CPU = False
REFRESH_CACHE = False

## Load LA protocol + readable audio

In [3]:
import random

protocol = LA_ROOT / PROTOCOL_BY_SPLIT[SPLIT]
records = read_la_cm_protocol(protocol)
print(f"Protocol {SPLIT}: {len(records)} utts")
print(f"Audio dir: {LA_ROOT / AUDIO_DIR_BY_SPLIT[SPLIT]}")

# LA protocols list bonafide first — stratified sample if capping
if MAX_UTTS > 0 and MAX_UTTS < len(records):
    rng = random.Random(42)
    bona = [r for r in records if r.label == 0]
    spoof = [r for r in records if r.label == 1]
    n_bona = max(1, MAX_UTTS // 2)
    n_spoof = MAX_UTTS - n_bona
    n_bona = min(n_bona, len(bona))
    n_spoof = min(n_spoof, len(spoof))
    records = rng.sample(bona, n_bona) + rng.sample(spoof, n_spoof)
    rng.shuffle(records)
    print(f"Stratified subset: {len(records)}")

cache_path = CACHE / f"la2019_{SPLIT}_readable.json"
readable, skipped = filter_readable_records(
    LA_ROOT,
    SPLIT,
    records,
    cache_path=cache_path,
    force_refresh=REFRESH_CACHE,
)

n_bona = sum(r.label == 0 for r in readable)
n_spoof = sum(r.label == 1 for r in readable)
print(f"Readable: {len(readable)} (bona={n_bona}, spoof={n_spoof}), skipped={len(skipped)}")
assert n_bona > 0 and n_spoof > 0, "Need both classes"

Protocol dev: 24844 utts
Audio dir: d:\speaker-verification-system\data\LA\ASVspoof2019_LA_dev\flac


LA dev readability:   0%|          | 0/24844 [00:00<?, ?it/s]

Readable: 24844 (bona=2548, spoof=22296), skipped=0


## Load mixed inverted-Mel checkpoint

In [4]:
device = torch.device("cpu" if FORCE_CPU or not torch.cuda.is_available() else "cuda")
ckpt = torch.load(CKPT, map_location=device, weights_only=False)
cfg = dict(ckpt["audio_config"])
if "feature_type" not in cfg:
    cfg["feature_type"] = ckpt.get("feature_type", "inverted_mel")
allowed = set(AudioConfig.__dataclass_fields__)
cfg = {k: v for k, v in cfg.items() if k in allowed}
config = AudioConfig(**cfg)
model = ReplayCNN(config).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()
train_thr = float(ckpt["threshold"])
print(f"feature={config.feature_type}  device={device}  train_thr={train_thr:.4f}")

feature=inverted_mel  device=cuda  train_thr=0.1546


## Score LA

In [5]:
ds = LAWaveformDataset(
    LA_ROOT,
    SPLIT,
    readable,
    config.sample_rate,
    config.samples,
    fix_length_fn=fix_length,
)
loader = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=device.type == "cuda",
)

@torch.inference_mode()
def collect_scores(model, loader, device):
    labels, scores, ids = [], [], []
    for waves, labs, batch_ids in tqdm(loader, desc="Scoring"):
        logits = model(waves.to(device, non_blocking=True))
        scores.extend(torch.sigmoid(logits).cpu().tolist())
        labels.extend(labs.tolist())
        ids.extend(batch_ids)
    return np.asarray(labels, dtype=int), np.asarray(scores, dtype=float), ids

labels, scores, utt_ids = collect_scores(model, loader, device)
print(f"Scored {len(labels)}  mean_score={scores.mean():.4f}")

Scoring:   0%|          | 0/3106 [00:00<?, ?it/s]

Scored 24844  mean_score=0.0948


## Metrics
Primary: **oracle EER on LA** (best threshold on this split). Also metrics at the mixed-train threshold.

In [6]:
def calculate_eer(y, s):
    live, spoof = s[y == 0], s[y == 1]
    if live.size == 0 or spoof.size == 0:
        raise ValueError("Need both classes")
    if float(live.max()) < float(spoof.min()):
        thr = (float(live.max()) + float(spoof.min())) / 2.0
        return 0.0, thr
    fpr, tpr, thresholds = roc_curve(y, s, pos_label=1)
    miss = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fpr - miss)))
    return float((fpr[idx] + miss[idx]) / 2.0), float(thresholds[idx])


def metrics_at(y, s, thr):
    preds = (s >= thr).astype(int)
    eer, eer_thr = calculate_eer(y, s)
    return {
        "threshold": float(thr),
        "eer": eer,
        "eer_percent": eer * 100.0,
        "eer_threshold_on_this_split": eer_thr,
        "accuracy": float(accuracy_score(y, preds)),
        "precision_spoof": float(precision_score(y, preds, zero_division=0)),
        "recall_spoof": float(recall_score(y, preds, zero_division=0)),
        "f1_spoof": float(f1_score(y, preds, zero_division=0)),
        "confusion_matrix_bona_spoof": confusion_matrix(y, preds, labels=[0, 1]).tolist(),
    }

eer, eer_thr = calculate_eer(labels, scores)
at_train = metrics_at(labels, scores, train_thr)
at_oracle = metrics_at(labels, scores, eer_thr)

summary = {
    "experiment": "inverted_mel_mixed_zero_shot_on_la2019",
    "checkpoint": str(CKPT),
    "la_root": str(LA_ROOT),
    "split": SPLIT,
    "n": int(len(labels)),
    "n_bonafide": int((labels == 0).sum()),
    "n_spoof": int((labels == 1).sum()),
    "feature_type": config.feature_type,
    "device": str(device),
    "train_threshold": train_thr,
    "oracle_eer_percent": eer * 100.0,
    "metrics_at_train_threshold": at_train,
    "metrics_at_oracle_eer_threshold": at_oracle,
}

md = f"""
| Setting | Value |
|---------|-------|
| Model | mixed inverted-Mel (2017+PA) |
| Target | ASVspoof 2019 LA `{SPLIT}` |
| N | {len(labels)} (bona={(labels==0).sum()}, spoof={(labels==1).sum()}) |
| **Oracle EER on LA** | **{eer*100:.2f}%** |
| Train-thr EER (same scores) | {at_train['eer_percent']:.2f}% |
| Acc @ train thr | {at_train['accuracy']*100:.1f}% |
| Acc @ oracle thr | {at_oracle['accuracy']*100:.1f}% |
"""
display(Markdown(md))
print(md)
print("confusion @ train thr", at_train["confusion_matrix_bona_spoof"])
print("confusion @ oracle thr", at_oracle["confusion_matrix_bona_spoof"])


| Setting | Value |
|---------|-------|
| Model | mixed inverted-Mel (2017+PA) |
| Target | ASVspoof 2019 LA `dev` |
| N | 24844 (bona=2548, spoof=22296) |
| **Oracle EER on LA** | **41.12%** |
| Train-thr EER (same scores) | 41.12% |
| Acc @ train thr | 27.8% |
| Acc @ oracle thr | 58.9% |



| Setting | Value |
|---------|-------|
| Model | mixed inverted-Mel (2017+PA) |
| Target | ASVspoof 2019 LA `dev` |
| N | 24844 (bona=2548, spoof=22296) |
| **Oracle EER on LA** | **41.12%** |
| Train-thr EER (same scores) | 41.12% |
| Acc @ train thr | 27.8% |
| Acc @ oracle thr | 58.9% |

confusion @ train thr [[2491, 57], [17878, 4418]]
confusion @ oracle thr [[1500, 1048], [9168, 13128]]


## Save artifacts

In [7]:
OUT.mkdir(parents=True, exist_ok=True)
(OUT / f"la2019_{SPLIT}_metrics.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
pred_path = OUT / f"la2019_{SPLIT}_predictions.csv"
with pred_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["utt_id", "true_label", "spoof_probability", "pred_at_train_thr"])
    preds = (scores >= train_thr).astype(int)
    for uid, lab, sc, pr in zip(utt_ids, labels, scores, preds):
        w.writerow([uid, "spoof" if lab else "bonafide", sc, "spoof" if pr else "bonafide"])

if skipped:
    (OUT / f"la2019_{SPLIT}_skipped_utts.txt").write_text(
        "\n".join(skipped) + "\n", encoding="utf-8"
    )

print("Wrote", OUT)
print("Oracle LA EER =", f"{eer*100:.2f}%")

Wrote d:\speaker-verification-system\replay-cnn-baseline\experiments\inverted_mel_mixed_on_la2019\runs\zero_shot_la
Oracle LA EER = 41.12%
